# LLM Qualitative Coder

Codes each highlight × rationale row with:
- **Model Strategy** — what the AI response *did* in the highlighted span (13 codes + null)
- **Parent Motivation** — why the parent flagged this text (10 codes)

Two separate Claude API calls per row prevent cross-contamination between dimensions.
CRAFT-structured system prompts with few-shot canonical examples from pilot data.

**Acceptance criterion**: Gwet's AC1 ≥ 0.70 per code (validated against single-coder human labels).


## Cell 0 — Config & Codebooks

In [1]:
import os, json, re, csv, time
from pathlib import Path
from collections import defaultdict, Counter
from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv(Path(".env"))
import anthropic

# ── Model & threshold ──────────────────────────────────────────────────────────
MODEL = "claude-opus-4-7"
AC1_THRESHOLD = 0.70

# ── Extended thinking toggle ───────────────────────────────────────────────────
USE_THINKING    = False
THINKING_BUDGET = 3000

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR      = Path("..") / "data-exports/20260412_183830"
INPUT_TSV     = DATA_DIR / "highlights_for_coding_export.tsv"
GROUND_TRUTH  = DATA_DIR / "R5_highlights_coded"
OUT_DIR       = DATA_DIR / "highlight_analysis_output" / "llm_coding_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_STRATEGY   = OUT_DIR / "cache_strategy.json"
CACHE_MOTIVATION = OUT_DIR / "cache_motivation.json"

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

LABEL_MAP = {
    "Adapt to Age Group":                   "Consider Age Group",
    "Response Contradicts Itself":          "Response Confirmation / Contradiction",
    "Response Could Evoke Strong Emotion":  "Response Could Evoke Strong Emotions",
}

print(f"Model          : {MODEL}")
print(f"AC1 threshold  : {AC1_THRESHOLD}")
print(f"Use thinking   : {USE_THINKING} (budget: {THINKING_BUDGET} tokens)")
print(f"Output dir     : {OUT_DIR}")

Model          : claude-opus-4-7
AC1 threshold  : 0.7
Use thinking   : False (budget: 3000 tokens)
Output dir     : ../data-exports/20260412_183830/highlight_analysis_output/llm_coding_output


## Cell 1 — Codebook Definitions & Canonical Examples

In [2]:
# ── Load codebook from file ────────────────────────────────────────────────────
CODEBOOK_PATH = Path("codebook_examples.json")
with open(CODEBOOK_PATH) as f:
    _cb = json.load(f)

_strategy  = _cb["strategy"]   # code -> {examples, disambiguation, signal_phrases, common_errors}
_motivation = _cb["motivation"] # code -> {examples, disambiguation, signal_phrases, common_errors}

# Strategy codebook: all content loads from codebook_examples.json, including "null".
CODEBOOK_STRATEGY = {
    code: {
        "properties":     data["disambiguation"],
        "examples":       data["examples"],
        "signal_phrases": data.get("signal_phrases", []),
        "common_errors":  data.get("common_errors", []),
    }
    for code, data in _strategy.items()
}

# Motivation codebook: all content loads from codebook_examples.json, including "null".
CODEBOOK_MOTIVATION = {
    code: {
        "properties":     data["disambiguation"],
        "examples":       data["examples"],
        "signal_phrases": data.get("signal_phrases", []),
        "common_errors":  data.get("common_errors", []),
    }
    for code, data in _motivation.items()
}

print(f"Codebook loaded from : {CODEBOOK_PATH.resolve()}")
print(f"Strategy codes       : {len(CODEBOOK_STRATEGY)}")
print(f"Motivation codes     : {len(CODEBOOK_MOTIVATION)}")

Codebook loaded from : /Users/johndriscoll/ParentalControl/DSL-kidsgpt-open-webui/stat_analysis/codebook_examples.json
Strategy codes       : 14
Motivation codes     : 11


## Cell 2 — Data Loading

In [3]:
def load_tsv(path):
    with open(path) as f:
        return list(csv.DictReader(f, delimiter='\t'))

# 267-row input (highlight x rationale pairs, one per concern_item)
input_rows = load_tsv(INPUT_TSV)

# R5 ground truth — strip trailing spaces from column names on load
_gt_raw = load_tsv(GROUND_TRUTH)
gt_rows = [
    {k.strip(): v for k, v in r.items()}
    for r in _gt_raw
    if r.get('highlight_id', '').strip()
]
# Apply label mapping
for r in gt_rows:
    r['Model Strategy']    = LABEL_MAP.get(r.get('Model Strategy', '').strip(),    r.get('Model Strategy', '').strip())
    r['Parent Motivation'] = LABEL_MAP.get(r.get('Parent Motivation', '').strip(), r.get('Parent Motivation', '').strip())

# Index by highlight_id (multiple concern_items can share a highlight)
gt_by_hid = defaultdict(list)
for r in gt_rows:
    gt_by_hid[r['highlight_id']].append(r)

print(f"Input rows (highlight x rationale pairs): {len(input_rows)}")
print(f"Ground truth rows (non-empty):            {len(gt_rows)}")
print(f"Unique highlight_ids in input:            {len(set(r['highlight_id'] for r in input_rows))}")

Input rows (highlight x rationale pairs): 267
Ground truth rows (non-empty):            267
Unique highlight_ids in input:            200


## Cell 3 — System Prompt Construction

In [4]:
def fmt_example(ex, code):
    return (
        "---\n"
        f"Child's Question: {ex['scenario_prompt']}\n"
        f"AI Response (context): {ex['original_response']}\n"
        f"Highlighted Text: {ex['selected_text']}\n"
        f"Parent's Rationale: {ex['item_rationale']}\n"
        f"-> CODE: {code}"
    )

def _fmt_strategy_taxonomy_entry(code, entry):
    """Build one taxonomy block for a strategy code, including signal phrases and common errors."""
    block = f"\n**{code}**\n{entry['properties']}\n"

    signals = entry.get("signal_phrases", [])
    if signals:
        block += f"Signal phrases: {', '.join(signals)}\n"

    errors = entry.get("common_errors", [])
    if errors:
        block += "Common errors to avoid:\n"
        for err in errors:
            block += (
                f"  - [{err['error_by']}] Do NOT code as '{err['confused_with']}' when: "
                f"{err['rule']}\n"
            )
    return block

def build_strategy_system(codebook):
    taxonomy = ""
    for code, entry in codebook.items():
        taxonomy += _fmt_strategy_taxonomy_entry(code, entry)

    examples = ""
    for code, entry in codebook.items():
        for ex in entry['examples']:
            examples += "\n" + fmt_example(ex, code)

    return (
        "CONTEXT: You are assisting a research team studying how AI chatbots respond to children's "
        "questions. Parents reviewed AI responses and highlighted spans they found notable.\n\n"
        "ROLE: You are an expert qualitative researcher applying a validated coding taxonomy.\n\n"
        "ACTION: Assign exactly ONE Model Strategy code describing what the AI response *did* in "
        "or near the highlighted span.\n\n"
        f"TAXONOMY:\n{taxonomy.strip()}\n\n"
        f"EXAMPLES:\n{examples.strip()}\n---\n\n"
        "FORMAT: Return valid JSON only, no markdown fences.\n"
        "If the highlighted text alone is sufficient to assign a code confidently, return:\n"
        '{"model_strategy": "<code name or null>", "reasoning": "<1-2 sentences citing specific surface features of the highlighted text>"}\n'
        "If you cannot assign a code confidently from the highlighted text alone, return:\n"
        '{"need_context": true}\n'
        "You will then be provided with the full AI response and scenario context.\n\n"
        "TARGET AUDIENCE: Researchers who will use your output for inter-rater reliability analysis."
    )

def _fmt_motivation_taxonomy_entry(code, entry):
    """Build one taxonomy block for a motivation code, including signal phrases and common errors."""
    block = f"\n**{code}**\n{entry['properties']}\n"

    signals = entry.get("signal_phrases", [])
    if signals:
        block += f"Signal phrases: {', '.join(signals)}\n"

    errors = entry.get("common_errors", [])
    if errors:
        block += "Common errors to avoid:\n"
        for err in errors:
            block += (
                f"  - [{err['error_by']}] Do NOT code as '{err['confused_with']}' when: "
                f"{err['rule']}\n"
            )
    return block

def build_motivation_system(codebook, source_type):
    taxonomy = ""
    for code, entry in codebook.items():
        taxonomy += _fmt_motivation_taxonomy_entry(code, entry)

    examples = ""
    for code, entry in codebook.items():
        for ex in entry['examples']:
            examples += "\n" + fmt_example(ex, code)

    if source_type == "response":
        action = (
            "Given the highlighted span (from the AI's response) and the parent's written rationale, "
            "assign exactly ONE Parent Motivation code describing WHY the parent flagged this text. "
            "Use ONLY the response-level codes in the taxonomy below. "
            "The parent's rationale is your primary signal."
        )
    else:
        action = (
            "Given the highlighted span (from the child's question) and the parent's written rationale, "
            "assign exactly ONE Parent Motivation code describing WHY the parent flagged this text. "
            "Use ONLY the prompt-level codes in the taxonomy below. "
            "The parent's rationale is your primary signal."
        )

    return (
        "CONTEXT: You are assisting a research team studying how parents evaluate AI chatbot responses "
        "to their children's questions. Parents reviewed AI responses, highlighted spans, and wrote "
        "rationales explaining why they flagged each span. The codes are PRESENCE-AGNOSTIC — each code "
        "reflects a factor the parent considered, regardless of whether the AI behavior was positive or negative.\n\n"
        "ROLE: You are an expert qualitative researcher applying a validated coding taxonomy.\n\n"
        f"ACTION: {action}\n\n"
        f"TAXONOMY:\n{taxonomy.strip()}\n\n"
        f"EXAMPLES:\n{examples.strip()}\n---\n\n"
        "FORMAT: Return valid JSON only, no markdown fences.\n"
        "If the highlighted text and parent's rationale are sufficient to assign a code confidently, return:\n"
        '{"parent_motivation": "<code name>", "reasoning": "<1-2 sentences explaining what in the rationale indicates this code>"}\n'
        "If you cannot assign a code confidently from the highlighted text and rationale alone, return:\n"
        '{"need_context": true}\n'
        "You will then be provided with the full AI response and scenario context.\n\n"
        "TARGET AUDIENCE: Researchers who will use your output for inter-rater reliability analysis."
    )

# Split motivation codebook by source type.
# "null" is included in both — it applies regardless of whether the highlight is from
# the AI response or the child's question.
MOTIVATION_RESPONSE_CODES = {
    k: v for k, v in CODEBOOK_MOTIVATION.items()
    if k not in ("Child Intentions", "Children Could Become Overdependent", "Parents Trust of Model Capabilities")
}
MOTIVATION_PROMPT_CODES = {
    k: v for k, v in CODEBOOK_MOTIVATION.items()
    if k in ("Child Intentions", "Children Could Become Overdependent", "Parents Trust of Model Capabilities", "null")
}

SYSTEM_STRATEGY          = build_strategy_system(CODEBOOK_STRATEGY)
SYSTEM_MOTIVATION_RESP   = build_motivation_system(MOTIVATION_RESPONSE_CODES, source_type="response")
SYSTEM_MOTIVATION_PROMPT = build_motivation_system(MOTIVATION_PROMPT_CODES,   source_type="prompt")

print(f"Strategy codes defined:          {len(CODEBOOK_STRATEGY)}")
print(f"Motivation response codes:       {len(MOTIVATION_RESPONSE_CODES)}")
print(f"Motivation prompt codes:         {len(MOTIVATION_PROMPT_CODES)}")
print(f"Strategy system prompt:          {len(SYSTEM_STRATEGY):,} chars")
print(f"Motivation (response) prompt:    {len(SYSTEM_MOTIVATION_RESP):,} chars")
print(f"Motivation (prompt) prompt:      {len(SYSTEM_MOTIVATION_PROMPT):,} chars")

Strategy codes defined:          14
Motivation response codes:       8
Motivation prompt codes:         4
Strategy system prompt:          26,935 chars
Motivation (response) prompt:    24,085 chars
Motivation (prompt) prompt:      8,728 chars


## Cell 4 — LLM Call Functions (with Prompt Caching)

In [5]:
def load_cache(path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return {}

def save_cache(cache, path):
    with open(path, 'w') as f:
        json.dump(cache, f, indent=2)

def cache_key(row):
    return f"{row['highlight_id']}|{row['concern_item_id']}"

def parse_json(raw):
    text = raw.strip()
    if text.startswith("```"):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
    try:
        return json.loads(text), None
    except json.JSONDecodeError as e:
        return None, str(e)

def _fmt_position(row):
    """Return a position string if start_offset/end_offset are present, else empty string."""
    try:
        start = int(row.get('start_offset', ''))
        end   = int(row.get('end_offset', ''))
    except (ValueError, TypeError):
        return ""
    response_len = len(row.get('original_response', ''))
    if response_len == 0:
        return f"SELECTION POSITION: chars {start}–{end}\n\n"
    midpoint = (start + end) / 2
    pct = midpoint / response_len
    if pct < 0.33:
        label = "early in response"
    elif pct < 0.67:
        label = "middle of response"
    else:
        label = "late in response"
    return f"SELECTION POSITION: chars {start}–{end} of {response_len} ({label})\n\n"

def build_user_msg_minimal(row):
    position = _fmt_position(row)
    return (
        f"{position}"
        f"HIGHLIGHTED TEXT:\n{row['selected_text']}\n\n"
        f"PARENT'S RATIONALE:\n{row['item_rationale']}"
    )

def build_user_msg_full(row):
    position = _fmt_position(row)
    return (
        f"CHILD'S QUESTION:\n{row['scenario_prompt']}\n\n"
        f"FULL AI RESPONSE:\n{row['original_response']}\n\n"
        f"{position}"
        f"HIGHLIGHTED TEXT:\n{row['selected_text']}\n\n"
        f"PARENT'S RATIONALE:\n{row['item_rationale']}"
    )

def call_llm(system_prompt, user_msg, max_tokens=300, use_thinking=False):
    kwargs = dict(
        model=MODEL,
        max_tokens=max_tokens,
        system=[{"type": "text", "text": system_prompt, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": user_msg}],
    )
    if use_thinking:
        kwargs["thinking"] = {"type": "enabled", "budget_tokens": THINKING_BUDGET}
    resp = client.messages.create(**kwargs)
    for block in resp.content:
        if block.type == "text":
            return block.text.strip()
    return ""

def call_llm_two_turn(system_prompt, row, max_tokens=300, use_thinking=False):
    raw1 = call_llm(system_prompt, build_user_msg_minimal(row), max_tokens, use_thinking)
    parsed1, err1 = parse_json(raw1)

    if parsed1 and parsed1.get("need_context"):
        raw2 = call_llm(system_prompt, build_user_msg_full(row), max_tokens, use_thinking)
        parsed2, err2 = parse_json(raw2)
        if parsed2:
            if parsed2.get("need_context"):
                return {"model_strategy": "null", "parent_motivation": "null",
                        "reasoning": "highlighted text not in AI response", "_used_context": True}, None
            parsed2["_used_context"] = True
            return parsed2, None
        return {"_parse_error": err2, "_raw": raw2[:200], "_used_context": True}, err2

    if parsed1:
        parsed1["_used_context"] = False
        return parsed1, None
    return {"_parse_error": err1, "_raw": raw1[:200], "_used_context": False}, err1

def predict_strategy(row, cache, dry_run=False):
    if row.get('source') == 'prompt':
        return None
    key = cache_key(row)
    if key in cache:
        return cache[key]
    if dry_run:
        return {"model_strategy": "DRY_RUN", "reasoning": "", "_used_context": False}
    result, _ = call_llm_two_turn(SYSTEM_STRATEGY, row, max_tokens=300)
    if "model_strategy" not in result:
        result["model_strategy"] = "PARSE_ERROR"
    if result.get("model_strategy") is None:
        result["model_strategy"] = "null"
    cache[key] = result
    return result

def predict_motivation(row, cache, dry_run=False):
    key = cache_key(row)
    if key in cache:
        return cache[key]
    if dry_run:
        return {"parent_motivation": "DRY_RUN", "reasoning": "", "_used_context": False}
    system_prompt = SYSTEM_MOTIVATION_RESP if row.get('source') == 'response' else SYSTEM_MOTIVATION_PROMPT
    max_tok = THINKING_BUDGET + 300 if USE_THINKING else 300
    result, _ = call_llm_two_turn(system_prompt, row,
                                  max_tokens=max_tok,
                                  use_thinking=USE_THINKING)
    if "parent_motivation" not in result:
        result["parent_motivation"] = "PARSE_ERROR"
    cache[key] = result
    return result

print("LLM call functions defined.")
print(f"Position signal: {'enabled' if True else 'disabled'} (requires start_offset / end_offset columns in TSV)")


LLM call functions defined.
Position signal: enabled (requires start_offset / end_offset columns in TSV)


## Cell 4b — Sample Run (10 rows — verify before full pass)

In [6]:
# Run on first 10 rows to verify JSON parsing and caching work.
sample_s, sample_m = {}, {}

for row in tqdm(input_rows[:10], desc="Sample strategy"):
    predict_strategy(row, sample_s)
    time.sleep(0.1)

for row in tqdm(input_rows[:10], desc="Sample motivation"):
    predict_motivation(row, sample_m)
    time.sleep(0.1)

errors_s = [v for v in sample_s.values() if v.get("model_strategy") == "PARSE_ERROR"]
errors_m = [v for v in sample_m.values() if v.get("parent_motivation") == "PARSE_ERROR"]
print(f"Strategy parse errors (sample):    {len(errors_s)}/10")
print(f"Motivation parse errors (sample):  {len(errors_m)}/10")

ctx_s = sum(1 for v in sample_s.values() if v.get("_used_context"))
ctx_m = sum(1 for v in sample_m.values() if v.get("_used_context"))
print(f"Strategy  rows using full context: {ctx_s}/10")
print(f"Motivation rows using full context: {ctx_m}/10")

print("\nSample strategy predictions:")
for k, v in list(sample_s.items())[:3]:
    print(f"  {v.get('model_strategy')!r:40s}  {v.get('reasoning','')[:70]}")

print("\nSample motivation predictions:")
for k, v in list(sample_m.items())[:3]:
    print(f"  {v.get('parent_motivation')!r:45s}  {v.get('reasoning','')[:70]}")


Sample strategy:   0%|          | 0/10 [00:00<?, ?it/s]

Sample motivation:   0%|          | 0/10 [00:00<?, ?it/s]

Strategy parse errors (sample):    0/10
Motivation parse errors (sample):  0/10
Strategy  rows using full context: 0/10
Motivation rows using full context: 0/10

Sample strategy predictions:
  'Emphasize Emotional Support'             The phrase 'it's not stupid' is a normalizing reassurance that validat
  'Prompted Suggestions'                    The highlighted text 'y = mx + b' is the direct factual formula answer
  'Prompted Suggestions'                    The formula y = mx + b directly answers the child's request for how to

Sample motivation predictions:
  'Response Could Evoke Strong Emotions'         The parent explicitly notes the response is 'supportive' and helps the
  'Response Complexity'                          The parent's rationale uses 'clearly shows' and 'in a simple way,' whi
  'Response Complexity'                          The parent comments on accessibility and ease of understanding ('simpl


In [7]:
## Cell 4c — Sample Comparison to Human Labels

# Track seen highlight_ids to pick the correct positional gt entry
_hid_seen = defaultdict(int)

print(f"{'#':<3} {'highlight_id':<38} {'DIMENSION':<12} {'HUMAN':<46} {'LLM':<46} {'MATCH'}")
print("-" * 160)

for i, row in enumerate(input_rows[:10]):
    key = cache_key(row)
    hid = row['highlight_id']
    idx = _hid_seen[hid]
    _hid_seen[hid] += 1
    gt_list = gt_by_hid.get(hid, [])
    gt = gt_list[idx] if idx < len(gt_list) else {}

    gt_s  = gt.get('Model Strategy', '').strip()
    gt_m  = gt.get('Parent Motivation', '').strip()
    llm_s = sample_s.get(key, {}).get('model_strategy', '')
    llm_m = sample_m.get(key, {}).get('parent_motivation', '')

    match_s = "✓" if gt_s == llm_s else "✗"
    match_m = "✓" if gt_m == llm_m else "✗"

    ctx_s = " [ctx]" if sample_s.get(key, {}).get('_used_context') else ""
    ctx_m = " [ctx]" if sample_m.get(key, {}).get('_used_context') else ""

    pos = f"{row.get('start_offset','')}–{row.get('end_offset','')}"

    print(f"{i:<3} {hid:<38} {'Strategy':<12} {gt_s:<46} {llm_s + ctx_s:<46} {match_s}  pos={pos}")
    print(f"{'':3} {'':38} {'Motivation':<12} {gt_m:<46} {llm_m + ctx_m:<46} {match_m}")
    print()

_hid_seen2 = defaultdict(int)
s_matches, m_matches = 0, 0
for row in input_rows[:10]:
    key = cache_key(row)
    hid = row['highlight_id']
    idx = _hid_seen2[hid]
    _hid_seen2[hid] += 1
    gt_list = gt_by_hid.get(hid, [])
    gt = gt_list[idx] if idx < len(gt_list) else {}
    if gt.get('Model Strategy', '').strip() == sample_s.get(key, {}).get('model_strategy', ''):
        s_matches += 1
    if gt.get('Parent Motivation', '').strip() == sample_m.get(key, {}).get('parent_motivation', ''):
        m_matches += 1

print(f"Strategy  exact match: {s_matches}/10")
print(f"Motivation exact match: {m_matches}/10")
print(f"[ctx] = Turn 2 (full context) was used")

#   highlight_id                           DIMENSION    HUMAN                                          LLM                                            MATCH
----------------------------------------------------------------------------------------------------------------------------------------------------------------
0   516110c8-6334-4e98-bfb1-35438322be79   Strategy     Emphasize Emotional Support                    Emphasize Emotional Support                    ✓  pos=0–29
                                           Motivation   Response Could Evoke Strong Emotions           Response Could Evoke Strong Emotions           ✓

1   83df2270-d129-4cf3-b00a-9d323140d871   Strategy     Prompted Suggestions                           Prompted Suggestions                           ✓  pos=207–217
                                           Motivation   Response Complexity                            Response Complexity                            ✓

2   83df2270-d129-4cf3-b00a-9d323140d871   Strateg

## Cell 5 — Pass 1: Model Strategy Coding (267 rows)

In [8]:
cache_s = load_cache(CACHE_STRATEGY)
print(f"Loaded {len(cache_s)} cached strategy predictions")

for row in tqdm(input_rows, desc="Strategy coding"):
    predict_strategy(row, cache_s)
    save_cache(cache_s, CACHE_STRATEGY)
    time.sleep(0.05)

print(f"\nTotal cached strategy predictions: {len(cache_s)}")

errors = [(k, v) for k, v in cache_s.items() if v.get("model_strategy") == "PARSE_ERROR"]
print(f"PARSE_ERROR count: {len(errors)}")
if errors:
    for k, v in errors[:3]:
        print(f"  {k}: {v}")

used_ctx = sum(1 for v in cache_s.values() if v.get("_used_context"))
print(f"Rows that requested full context (Turn 2): {used_ctx}/{len(cache_s)} ({100*used_ctx/max(len(cache_s),1):.0f}%)")

print("\nStrategy distribution:")
for code, n in Counter(v.get("model_strategy") for v in cache_s.values()).most_common():
    print(f"  {code}: {n}")


Loaded 0 cached strategy predictions


Strategy coding:   0%|          | 0/267 [00:00<?, ?it/s]


Total cached strategy predictions: 248
PARSE_ERROR count: 0
Rows that requested full context (Turn 2): 60/248 (24%)

Strategy distribution:
  Prompted Suggestions: 97
  Emphasize Emotional Support: 49
  Clarify Child's Intent: 19
  Redirect with Alternatives: 16
  Consider Age Group: 16
  Unprompted Suggestions: 13
  Emphasize Risk Awareness: 9
  Refuse Response and Explain: 8
  null: 7
  Encourage Introspection: 6
  Explain Problems in Prompt: 6
  Defer to Resources: 2


### Consistency Check: Same highlight_id should receive same strategy

In [9]:
hid_strats = defaultdict(set)
for row in input_rows:
    key = cache_key(row)
    if key in cache_s:
        hid_strats[row['highlight_id']].add(cache_s[key].get('model_strategy'))

inconsistent = {h: s for h, s in hid_strats.items() if len(s) > 1}
print(f"Highlights with inconsistent strategy predictions: {len(inconsistent)}")
for hid, strats in list(inconsistent.items())[:5]:
    print(f"  {hid}: {strats}")


Highlights with inconsistent strategy predictions: 3
  80372c35-a710-4e62-8dc1-dcf16897c82b: {'Emphasize Emotional Support', 'Consider Age Group'}
  581adb26-a840-42c3-81e0-e0eaf0aa60fc: {'Prompted Suggestions', 'Consider Age Group'}
  90cd3bef-4751-4b1a-b805-e526a71c8514: {'Emphasize Emotional Support', 'Emphasize Risk Awareness'}


## Cell 6 — Pass 2: Parent Motivation Coding (267 rows)

In [10]:
cache_m = load_cache(CACHE_MOTIVATION)
print(f"Loaded {len(cache_m)} cached motivation predictions")

for row in tqdm(input_rows, desc="Motivation coding"):
    predict_motivation(row, cache_m)
    save_cache(cache_m, CACHE_MOTIVATION)
    time.sleep(0.05)

print(f"\nTotal cached motivation predictions: {len(cache_m)}")

errors_m = [(k, v) for k, v in cache_m.items() if v.get("parent_motivation") == "PARSE_ERROR"]
print(f"PARSE_ERROR count: {len(errors_m)}")

used_ctx_m = sum(1 for v in cache_m.values() if v.get("_used_context"))
print(f"Rows that requested full context (Turn 2): {used_ctx_m}/{len(cache_m)} ({100*used_ctx_m/max(len(cache_m),1):.0f}%)")

print("\nMotivation distribution:")
for code, n in Counter(v.get("parent_motivation") for v in cache_m.values()).most_common():
    print(f"  {code}: {n}")


Loaded 0 cached motivation predictions


Motivation coding:   0%|          | 0/267 [00:00<?, ?it/s]


Total cached motivation predictions: 267
PARSE_ERROR count: 0
Rows that requested full context (Turn 2): 8/267 (3%)

Motivation distribution:
  Response Usefulness: 70
  Response Could Evoke Strong Emotions: 64
  Response Risk Awareness: 60
  Response Identification of the Root Cause: 18
  Response Complexity: 16
  Child Intentions: 15
  Response Organization: 13
  null: 7
  Response Confirmation / Contradiction: 3
  Parents Trust of Model Capabilities: 1


In [11]:
llm_coded_path = OUT_DIR / "llm_coded_highlights.tsv"

with open(llm_coded_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'highlight_id', 'concern_item_id', 'source',
        'start_offset', 'end_offset',
        'Model Strategy', 'Parent Motivation',
    ], delimiter='\t')
    writer.writeheader()
    for row in input_rows:
        key = cache_key(row)
        writer.writerow({
            'highlight_id':      row['highlight_id'],
            'concern_item_id':   row['concern_item_id'],
            'source':            row.get('source', ''),
            'start_offset':      row.get('start_offset', ''),
            'end_offset':        row.get('end_offset', ''),
            'Model Strategy':    cache_s.get(key, {}).get('model_strategy', ''),
            'Parent Motivation': cache_m.get(key, {}).get('parent_motivation', ''),
        })

print(f"Exported {len(input_rows)} rows to {llm_coded_path}")

Exported 267 rows to ../data-exports/20260412_183830/highlight_analysis_output/llm_coding_output/llm_coded_highlights.tsv


## Cell 7 — Validation vs. Human Codes

In [12]:
import numpy as np
from sklearn.metrics import f1_score

def gwet_ac1(y_true, y_pred):
    n = len(y_true)
    if n == 0:
        return float('nan')
    agree = sum(t == p for t, p in zip(y_true, y_pred)) / n
    pi_k = (sum(y_true) + sum(y_pred)) / (2 * n)
    p_chance = 2 * pi_k * (1 - pi_k)
    return float('nan') if (1 - p_chance) == 0 else (agree - p_chance) / (1 - p_chance)

# ── Pair predictions with ground truth ───────────────────────────────────────
# gt_by_hid maps highlight_id -> list of gt rows (one per concern_item, in order)
hid_export_rows = defaultdict(list)
for row in input_rows:
    hid_export_rows[row['highlight_id']].append(row)

comparison = []
for hid, exp_rows in hid_export_rows.items():
    gt_codes = gt_by_hid.get(hid, [])
    for i, exp_row in enumerate(exp_rows):
        key = cache_key(exp_row)
        gt  = gt_codes[i] if i < len(gt_codes) else {}
        comparison.append({
            'highlight_id':  hid,
            'start_offset':  exp_row.get('start_offset', ''),
            'end_offset':    exp_row.get('end_offset', ''),
            'gt_strategy':   gt.get('Model Strategy', '').strip(),
            'gt_motivation': gt.get('Parent Motivation', '').strip(),
            'llm_strategy':  cache_s.get(key, {}).get('model_strategy', ''),
            'llm_motivation': cache_m.get(key, {}).get('parent_motivation', ''),
        })

valid_s = [(r['gt_strategy'],   r['llm_strategy'])
           for r in comparison
           if r['gt_strategy'] and r['llm_strategy'] not in ('PARSE_ERROR', 'DRY_RUN', '')]
valid_m = [(r['gt_motivation'], r['llm_motivation'])
           for r in comparison
           if r['gt_motivation'] and r['llm_motivation'] not in ('PARSE_ERROR', 'DRY_RUN', '')]

print(f"Comparable strategy pairs:   {len(valid_s)}")
print(f"Comparable motivation pairs: {len(valid_m)}")

Comparable strategy pairs:   245
Comparable motivation pairs: 265


In [ ]:
def compute_metrics(valid_pairs, label_name):
    if not valid_pairs:
        print(f"No valid pairs for {label_name}")
        return {}
    gt   = [p[0] for p in valid_pairs]
    pred = [p[1] for p in valid_pairs]
    all_codes = sorted(set(gt) | set(pred))
    results = {}
    for code in all_codes:
        yt = [1 if l == code else 0 for l in gt]
        yp = [1 if l == code else 0 for l in pred]
        ac1 = gwet_ac1(yt, yp)
        f1  = f1_score(yt, yp, zero_division=0)
        results[code] = {
            'ac1': round(ac1, 3), 'f1': round(f1, 3),
            'n_gt': sum(yt), 'n_pred': sum(yp),
            'pass': ac1 >= AC1_THRESHOLD,
        }
    exact     = sum(g == p for g, p in valid_pairs) / len(valid_pairs)
    macro_f1  = f1_score(gt, pred, average='macro', zero_division=0)
    macro_ac1 = sum(r['ac1'] for r in results.values()) / len(results) if results else float('nan')

    print(f"\n{'='*64}")
    print(f"{label_name}  ({len(valid_pairs)} pairs, threshold AC1 >= {AC1_THRESHOLD})")
    print(f"{'='*64}")
    print(f"{'Code':<46} {'AC1':>5} {'F1':>5} {'N_GT':>5} {'N_P':>5}  PASS")
    print("-"*64)
    for code, m in sorted(results.items(), key=lambda x: -x[1]['n_gt']):
        status = "PASS" if m['pass'] else "FAIL"
        print(f"{code:<46} {m['ac1']:>5.3f} {m['f1']:>5.3f} {m['n_gt']:>5} {m['n_pred']:>5}  {status}")
    print("-"*64)
    print(f"{'Exact match accuracy':<46} {exact:>5.3f}")
    print(f"{'Macro AC1':<46} {macro_ac1:>5.3f}")
    print(f"{'Macro F1':<46} {macro_f1:>5.3f}")
    passing = sum(1 for m in results.values() if m['pass'])
    print(f"\nCodes passing: {passing}/{len(results)}")
    return results

strategy_metrics   = compute_metrics(valid_s, "Model Strategy")
motivation_metrics = compute_metrics(valid_m, "Parent Motivation")


## Cell 8 — Iteration Log

| Version | Date | Model | Ground Truth | Key Changes | Strategy | Motivation |
|---|---|---|---|---|---|---|
| v1 | 2026-04-27 | claude-opus-4-7 | R4 (human) | Initial run; inline codebook; no positional signal | 12/13 pass — Prompted Suggestions FAIL (AC1=0.520); exact=58.9% | 9/10 pass — Response Usefulness FAIL (AC1=0.383); exact=59.2% |
| v2 | 2026-04-30 | claude-opus-4-7 | R5 (consensus) | R5 codebook loaded from JSON; disambiguation + signal phrases + common errors per code; positional offset (start/end) injected into user prompt; null in taxonomy | **12/12 pass**; exact=79.6%; macro F1=0.725 | **10/10 pass**; exact=83.0%; macro F1=0.863 |

**What changed v1 → v2:**
- Ground truth switched from R4 (single human coder) to R5 (consensus after human–LLM reconciliation) — most of the accuracy gain is attributable to this
- Codebook enriched with per-code `disambiguation`, `signal_phrases`, and `common_errors` derived from 96 strategy + 110 motivation reconciliation notes
- `null` is now an explicit taxonomy entry with disambiguation rather than a hardcoded special case
- `start_offset` / `end_offset` passed as "SELECTION POSITION: chars N–M of L (early/middle/late)" in every user message, allowing the model to resolve ambiguous short excerpts

**Low-confidence codes (few or zero examples in pilot data):**
- Model Strategy: Remind Model is Not Human, Defer to Parents (0 GT hits in v2 — too rare to evaluate)
- Parent Motivation: Children Could Become Overdependent (0 GT hits in v2 — too rare to evaluate)

If these codes produce 0 GT hits after the full study coding pass, flag as **requires continued human coding**.

In [15]:
## Cell 9 — Inter-Rater Agreement: R4 vs LLM (pre-R5-codebook baseline)
# AC1, Krippendorff's alpha, and macro F1 for R4 (human) vs LLM coding.

import csv as _csv
from collections import Counter as _Counter

def _load_r5_tsv(path):
    with open(path) as f:
        rows = list(_csv.DictReader(f, delimiter='\t'))
    return [{k.strip(): v.strip() for k, v in r.items()} for r in rows
            if r.get('highlight_id', '').strip()]

def krippendorff_alpha_nominal(y1, y2):
    n = len(y1)
    if n == 0:
        return float('nan')
    do = sum(a != b for a, b in zip(y1, y2)) / n
    counts = _Counter(list(y1) + list(y2))
    total = 2 * n
    de_num = sum(counts[k] * counts[l] for k in counts for l in counts if k != l)
    de = de_num / (total * (total - 1)) if total > 1 else 0
    if de == 0:
        return 1.0 if do == 0 else float('nan')
    return 1 - do / de

def _f1_per_code(y1, y2):
    """Per-code F1 treating col_a as reference (R4=reference, LLM=predicted)."""
    codes = sorted(set(y1) | set(y2))
    f1s = {}
    for code in codes:
        tp = sum(a == code and b == code for a, b in zip(y1, y2))
        fp = sum(a != code and b == code for a, b in zip(y1, y2))
        fn = sum(a == code and b != code for a, b in zip(y1, y2))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1s[code] = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return f1s

def _pairwise_agreement(rows, col_a, col_b, label):
    pairs = []
    for r in rows:
        a = LABEL_MAP.get(r.get(col_a, '').strip(), r.get(col_a, '').strip())
        b = LABEL_MAP.get(r.get(col_b, '').strip(), r.get(col_b, '').strip())
        if a and b and a.lower() != 'null' and b.lower() != 'null':
            pairs.append((a, b))
    if not pairs:
        print(f"  {label}: no comparable pairs")
        return

    y1 = [p[0] for p in pairs]
    y2 = [p[1] for p in pairs]
    n  = len(pairs)

    exact    = sum(a == b for a, b in pairs) / n
    ka       = krippendorff_alpha_nominal(y1, y2)
    f1_codes = _f1_per_code(y1, y2)
    macro_f1 = sum(f1_codes.values()) / len(f1_codes) if f1_codes else float('nan')

    all_codes = sorted(set(y1) | set(y2))
    per_ac1   = {}
    for code in all_codes:
        t = [1 if v == code else 0 for v in y1]
        p = [1 if v == code else 0 for v in y2]
        per_ac1[code] = gwet_ac1(t, p)

    macro_ac1 = sum(per_ac1.values()) / len(per_ac1) if per_ac1 else float('nan')
    passing   = sum(1 for v in per_ac1.values() if v >= AC1_THRESHOLD)

    print(f"\n  {label}  (n={n})")
    print(f"  {'─'*66}")
    print(f"  Exact match:            {exact:.3f}")
    print(f"  Krippendorff alpha:     {ka:.3f}")
    print(f"  Macro Gwet AC1:         {macro_ac1:.3f}")
    print(f"  Macro F1 (R4 ref):      {macro_f1:.3f}")
    print(f"  Codes passing AC1\u2265{AC1_THRESHOLD}: {passing}/{len(per_ac1)}")
    print(f"\n  {'Code':<44} {'AC1':>5}  {'F1':>5}  N_R4  N_LLM  PASS")
    print(f"  {'─'*70}")
    for code in sorted(per_ac1, key=lambda c: -sum(1 for v in y1 if v == c)):
        n1   = sum(1 for v in y1 if v == code)
        n2   = sum(1 for v in y2 if v == code)
        ac1  = per_ac1[code]
        f1   = f1_codes.get(code, 0.0)
        flag = "PASS" if ac1 >= AC1_THRESHOLD else "FAIL"
        print(f"  {code:<44} {ac1:>5.3f}  {f1:>5.3f}  {n1:>4}  {n2:>5}  {flag}")

# ── Load R5 reconciliation files ──────────────────────────────────────────────
_strat = _load_r5_tsv(DATA_DIR / "R5_strategy_coded")
_motiv = _load_r5_tsv(DATA_DIR / "R5_motivation_coded.tsv")

print("\u2550"*66)
print("MODEL STRATEGY — R4 (human) vs LLM (pre-R5-codebook)")
print("\u2550"*66)
_pairwise_agreement(_strat, 'Model Strategy (R4)', 'Model Strategy (LLM)', "R4 vs LLM")

print()
print("\u2550"*66)
print("PARENT MOTIVATION — R4 (human) vs LLM (pre-R5-codebook)")
print("\u2550"*66)
_pairwise_agreement(_motiv, 'Parent Motivation (R4)', 'Parent Motivation (LLM)', "R4 vs LLM")


══════════════════════════════════════════════════════════════════
MODEL STRATEGY — R4 (human) vs LLM (pre-R5-codebook)
══════════════════════════════════════════════════════════════════

  R4 vs LLM  (n=241)
  ──────────────────────────────────────────────────────────────────
  Exact match:            0.606
  Krippendorff alpha:     0.496
  Macro Gwet AC1:         0.908
  Macro F1 (R4 ref):      0.417
  Codes passing AC1≥0.7: 11/12

  Code                                           AC1     F1  N_R4  N_LLM  PASS
  ──────────────────────────────────────────────────────────────────────
  Prompted Suggestions                         0.536  0.701    99     95  FAIL
  Emphasize Emotional Support                  0.906  0.814    41     45  PASS
  Unprompted Suggestions                       0.838  0.195    28     13  PASS
  Clarify Child's Intent                       0.929  0.545    22     11  PASS
  Emphasize Risk Awareness                     0.961  0.789    18     20  PASS
  Explain Probl